# Q12.
```{admonition}
:class: note
Write a function to implement matrix completion that makes use of `PCA()` rather than `svd()`.

In [1]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

In [2]:
def matrix_completion(X:np.array,max_iter = 100, tolerance=1e-5,M=1):
    missing = np.isnan(X)
    nan_idx = np.where(missing)
    completed_X = X.copy()
    completed_X[nan_idx] = np.nanmean(completed_X,axis=0)[nan_idx[1]]

    iteration = 0
    rel_error = 1
    mss0 = np.mean(X[~missing]**2)
    mss_old = np.mean(completed_X[~missing]**2)
    
    for k in range(max_iter):
        iteration += 1
        pca = PCA(n_components=M)
        z = pca.fit_transform(completed_X)
        phi = pca.components_
        X_approx = z@phi
        completed_X[missing] = X_approx[missing]
        mss = mean_squared_error(completed_X,X_approx) #Missing data points are subtracted to 0 and do not contribute to sum
        rel_error = np.abs((mss_old-mss)/mss0)
        mss_old = mss
        if rel_error < tolerance:
            break

    return completed_X